In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score

import lightgbm as lgb

RANDOM_STATE = 42

app = pd.read_csv("../data/raw/application_train.csv")

X = app.drop(columns=["TARGET", "SK_ID_CURR"])
y = app["TARGET"]

X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("Development set:", X_dev.shape)
print("Test set:", X_test.shape)
print("Development default rate:", round(y_dev.mean(), 4))
print("Test default rate:", round(y_test.mean(), 4))

Development set: (246008, 120)
Test set: (61503, 120)
Development default rate: 0.0807
Test default rate: 0.0807


In [3]:
def basic_clean(df):
    df = df.copy()

    df["DAYS_EMPLOYED_ANOM"] = (
        df["DAYS_EMPLOYED"] == 365243
    ).astype(int)

    df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(
        365243, np.nan
    )

    return df


X_dev_c = basic_clean(X_dev)
X_test_c = basic_clean(X_test)

print("Original columns:", X_dev.shape[1])
print("Cleaned columns:", X_dev_c.shape[1])
print(
    "Anomaly count:",
    X_dev_c["DAYS_EMPLOYED_ANOM"].sum()
)

Original columns: 120
Cleaned columns: 121
Anomaly count: 44143


In [4]:
num_cols = X_dev_c.select_dtypes(include="number").columns.tolist()
cat_cols = X_dev_c.select_dtypes(exclude="number").columns.tolist()

print("Numeric columns:", len(num_cols))
print("Categorical columns:", len(cat_cols))

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(
            strategy="median",
            add_indicator=True
        )),
        ("scale", StandardScaler())
    ]), num_cols),

    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=100,
            sparse_output=True
        ))
    ]), cat_cols),
])

Numeric columns: 105
Categorical columns: 16


In [5]:
results = []


def score(name, model, X, y):
    aucs = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="roc_auc",
        n_jobs=1
    )

    results.append({
        "model": name,
        "cv_auc": aucs.mean(),
        "cv_std": aucs.std()
    })

    print(f"{name}: {aucs.mean():.4f} ± {aucs.std():.4f}")


score(
    "Baseline (constant)",
    DummyClassifier(
        strategy="stratified",
        random_state=RANDOM_STATE
    ),
    X_dev_c,
    y_dev
)


score(
    "Logistic Regression",
    Pipeline([
        ("prep", preprocessor),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced"
        ))
    ]),
    X_dev_c,
    y_dev
)

Baseline (constant): 0.5003 ± 0.0012
Logistic Regression: 0.7474 ± 0.0016


In [6]:
def prep_for_lgbm(df):
    df = df.copy()

    # Preserve the information carried by the DAYS_EMPLOYED placeholder
    df["DAYS_EMPLOYED_ANOM"] = (
        df["DAYS_EMPLOYED"] == 365243
    ).astype("int8")

    # Replace the placeholder with NaN
    df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(
        365243, np.nan
    )

    # Convert categorical columns to LightGBM's native category dtype
    for col in df.select_dtypes(exclude="number").columns:
        df[col] = df[col].astype("category")

    # Reduce numerical memory usage
    for col in df.select_dtypes(include="float64").columns:
        df[col] = df[col].astype("float32")

    return df


X_dev_lgb = prep_for_lgbm(X_dev)

print("Shape:", X_dev_lgb.shape)
print("Memory:", round(
    X_dev_lgb.memory_usage(deep=True).sum() / 1e6, 2
), "MB")

print("\nCategorical columns:",
      X_dev_lgb.select_dtypes(include="category").shape[1])

print("Numeric columns:",
      X_dev_lgb.select_dtypes(include="number").shape[1])

Shape: (246008, 121)
Memory: 145.89 MB

Categorical columns: 16
Numeric columns: 105


In [7]:
model = lgb.LGBMClassifier(
    random_state=RANDOM_STATE,
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    verbose=-1,
    n_jobs=2,
)

aucs = cross_val_score(
    model,
    X_dev_lgb,
    y_dev,
    cv=cv,
    scoring="roc_auc",
    n_jobs=1
)

print(f"LightGBM: {aucs.mean():.4f} ± {aucs.std():.4f}")

results.append({
    "model": "LightGBM",
    "cv_auc": aucs.mean(),
    "cv_std": aucs.std()
})

LightGBM: 0.7536 ± 0.0019


In [8]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_dev_lgb,
    y_dev,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_dev
)

m = lgb.LGBMClassifier(
    random_state=RANDOM_STATE,
    n_estimators=300,
    learning_rate=0.05,
    verbose=-1,
    n_jobs=2
)

m.fit(X_tr, y_tr)

probs = m.predict_proba(X_val)[:, 1]

rows = []

for rate in [0.05, 0.10, 0.20, 0.30, 0.50]:
    n_reject = int(len(probs) * rate)

    riskiest = np.argsort(probs)[-n_reject:]

    caught = y_val.iloc[riskiest].sum()

    rows.append({
        "reject_rate": rate,
        "defaults_caught": caught / y_val.sum(),
        "good_customers_rejected": (
            n_reject - caught
        ) / (len(y_val) - y_val.sum())
    })

capture = pd.DataFrame(rows)

capture

,reject_rate,defaults_caught,good_customers_rejected
0,0.05,0.200000,0.036825
1,0.10,0.326284,0.080125
2,0.20,0.514602,0.172365
3,0.30,0.643907,0.269788
4,0.50,0.818933,0.471992
